In [57]:
from pathlib import Path
import numpy as np
import pandas as pd
import tomotopy as tp
from tomotopy.utils import Corpus
from tomotopy.coherence import Coherence
import matplotlib.pyplot as plt
import seaborn as sns
import pyLDAvis

In [58]:
df = pd.read_csv("cleaned_corpus.csv", index_col = 0, parse_dates = ['approved_date'])
df.info()

<class 'pandas.DataFrame'>
Index: 994 entries, repacts:datasets/markdown/repacts/2016/ra_10923_2016.md to repacts:datasets/markdown/repacts/2022/ra_11933_2022.md
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   source           994 non-null    str           
 1   year             994 non-null    int64         
 2   month            994 non-null    str           
 3   path             994 non-null    str           
 4   basename         994 non-null    str           
 5   title            994 non-null    str           
 6   day              994 non-null    int64         
 7   year_extracted   994 non-null    int64         
 8   approved_date    994 non-null    datetime64[us]
 9   cleaned_content  994 non-null    str           
dtypes: datetime64[us](1), int64(3), str(6)
memory usage: 6.2 MB


In [59]:
df['quarter'] = df.approved_date.dt.quarter
# q1 = df[df.quarters == 1]
# q2 = df[df.quarters == 2]
# q3 = df[df.quarters == 3]
# q4 = df[df.quarters == 4]

In [60]:
years = set(df['year'])
quarters = set(df['quarter'])
models = {}
for y in years:
    for q in quarters:
        # collect into corpus obj
        corpus = Corpus()
        period = df[(df.year == y) & (df.quarter == q)] # pyright: ignore[reportUnknownVariableType]
        print(f"found {len(period)} records for year {y} quarter {q}")
        if len(period):
            for text in period['cleaned_content']: # pyright: ignore[reportUnknownVariableType]
                corpus.add_doc(text.split()) # pyright: ignore[reportUnknownArgumentType, reportUnknownMemberType]

            # training the model
            seed = 357
            # k = 5 topics
            model = tp.LDAModel(k = 5, corpus = corpus, seed = seed)
            iters = 1000
            model.train(iterations=iters, workers=1) # pyright: ignore[reportCallIssue, reportUnknownMemberType]
            models[(y, q)] = model
        else:
            print("skipping, no records")

len(models)

found 0 records for year 2016 quarter 1
skipping, no records
found 0 records for year 2016 quarter 2
skipping, no records
found 0 records for year 2016 quarter 3
skipping, no records
found 1 records for year 2016 quarter 4
found 0 records for year 2017 quarter 1
skipping, no records
found 2 records for year 2017 quarter 2
found 24 records for year 2017 quarter 3
found 14 records for year 2017 quarter 4
found 45 records for year 2018 quarter 1
found 42 records for year 2018 quarter 2
found 32 records for year 2018 quarter 3
found 81 records for year 2018 quarter 4
found 82 records for year 2019 quarter 1
found 86 records for year 2019 quarter 2
found 116 records for year 2019 quarter 3
found 3 records for year 2019 quarter 4
found 4 records for year 2020 quarter 1
found 8 records for year 2020 quarter 2
found 17 records for year 2020 quarter 3
found 25 records for year 2020 quarter 4
found 14 records for year 2021 quarter 1
found 35 records for year 2021 quarter 2
found 22 records for y

23

In [ ]:
def top_words(model, k):
    """Print the top words for topic k in a model."""
    top_words = model.get_topic_words(topic_id = k, top_n = 5)
    top_words = [f"{word} ({score:0.4f}%)" for (word, score) in top_words]
    print(f"Topic {k}: {', '.join(top_words)}")

for k,model in models.items(): # pyright: ignore[reportUnknownVariableType]
    year, quarter = k # pyright: ignore[reportUnknownVariableType]
    print(f"topics for year {year} quarter {quarter}")
    for i in range(model.k): # pyright: ignore[reportUnknownArgumentType, reportUnknownMemberType]
        top_words(model, i) # pyright: ignore[reportUnknownArgumentType]

# TODO: 'amend' meron pa rin? to check the rest of the results

topics for year 2016 quarter 4
Topic 0: official (0.0783%), regulation (0.0471%), provide (0.0471%), five (0.0471%), employee (0.0314%)
Topic 1: sangguniang (0.1937%), amend (0.0862%), thousand (0.0539%), use (0.0431%), house (0.0431%)
Topic 2: election (0.2237%), barangay (0.1767%), million (0.0472%), provision (0.0472%), immediately (0.0354%)
Topic 3: kabataan (0.1772%), hundred (0.0835%), hold (0.0522%), two (0.0418%), senate (0.0418%)
Topic 4: october (0.0557%), appropriation (0.0557%), peso (0.0446%), representative (0.0446%), follow (0.0446%)
topics for year 2017 quarter 2
Topic 0: franchise (0.1445%), use (0.0347%), law (0.0318%), inc (0.0289%), year (0.0260%)
Topic 1: service (0.0565%), grant (0.0509%), provide (0.0452%), system (0.0311%), senate (0.0311%)
Topic 2: public (0.0618%), representative (0.0450%), house (0.0422%), congress (0.0393%), broadcast (0.0337%)
Topic 3: grantee (0.1299%), telecommunication (0.0392%), follow (0.0343%), communication (0.0221%), successor (0.02